# CS781 Capstone Phase 7: Prior Case Retrieval (PCR)
## Submission 07: Poly-Encoder Architecture for Multi-Vector Late Interaction (~154.2M)

* **Task**: Prior Case Retrieval (PCR) — Retrieve Top 7 legally & semantically relevant prior cases.
* **Target Metric**: **Micro-F1@7** (Secondary: Micro-Precision@7, Micro-Recall@7).
* **Base Foundation Model**: `sarweshbarnwal/CS781-Legal-Models` -> `model4-capacity-boundary-154m` (14 layers, $d=896$, GQA 4:1, Tied Embeddings).
* **Poly-Encoder Architecture ([Humeau et al., ICLR 2020](https://arxiv.org/abs/1905.01969))**:
  * **Multi-Vector Query Representation**: Instead of compressing an entire legal case into a single monolithic embedding, the query case is projected into **$M=8$ learnable latent aspect vectors** ($C \in \mathbb{R}^{8 \times 896}$):
    $$w_{m} = \text{Softmax}\left(\frac{c_m \cdot H^T}{\sqrt{d}}\right), \quad y_m = \sum_{i} w_{m, i} h_i \quad (m \in [1, \dots, 8])$$
    Each codebook vector $c_m$ specializes in extracting a distinct legal aspect: ratio decidendi, statutory provisions, procedural posture, evidentiary facts, relief sought, precedent citations, etc.
  * **Candidate Pre-Encoding**: Each candidate case is encoded into a single dense L2-normalized vector $v \in \mathbb{R}^{896}$, preserving $O(N)$ offline pre-computation.
  * **Late-Interaction Attention**: During scoring, candidate vector $v$ attends over the query's 8 aspect vectors:
    $$\alpha_m = \text{Softmax}\left(\frac{v \cdot y_m}{\sqrt{d}}\right), \quad u(v) = \sum_{m=1}^8 \alpha_m y_m$$
    $$s(q, v) = u(v) \cdot v$$
    Yields cross-encoder expressive power with bi-encoder retrieval speed (<1ms per query).
* **Loss Objective**: InfoNCE / Multiple Negatives Ranking Loss ($\tau = 0.05$) across in-batch negatives.
* **Dual-T4 Acceleration**: PyTorch DataParallel across `cuda:0` and `cuda:1` for training and candidate pre-encoding.
* **Continuous Cloud Sync**: Push checkpoints every 200 steps to `sarweshbarnwal/CS781-Legal-Models/Phase_7/sub07_model4_poly_encoder_multiaspect/` with automatic crash recovery.
* **Strict CodaBench Output**: Verify exactly 7 unique candidate IDs per query from `test_candidates.json`, output root `submission.zip`.


In [ ]:
# 1. Environment Dependencies & Hugging Face Authentication
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import sys
import gc
import json
import time
import math
import glob
import random
import zipfile
import shutil
import threading
from typing import List, Dict, Tuple, Optional, Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Defensive PyTorch & Transformers compatibility guard for Python 3.13 / Kaggle
import sys
import types
import importlib
import torch

try:
    _u = importlib.import_module("torch._utils")
except Exception:
    _u = types.ModuleType("torch._utils")

if not hasattr(_u, "_get_device_index"):
    def _get_device_index(device, optional=False, allow_cpu=False):
        if isinstance(device, str): device = torch.device(device)
        if isinstance(device, torch.device): return -1 if device.type == "cpu" else (device.index or 0)
        if isinstance(device, int): return device
        return 0
    _u._get_device_index = _get_device_index

if not hasattr(_u, "try_import"):
    def try_import(name):
        try:
            return importlib.import_module(name)
        except Exception:
            return None
    _u.try_import = try_import

def _fallback_getattr(name):
    if name == "try_import":
        return _u.try_import
    if name == "_get_device_index":
        return _u._get_device_index
    return lambda *args, **kwargs: None

_u.__getattr__ = _fallback_getattr
setattr(torch, "_utils", _u)
sys.modules["torch._utils"] = _u

try:
    import transformers.utils.import_utils as hf_import_utils
    hf_import_utils.is_torchvision_available = lambda: False
except Exception:
    pass

try:
    import transformers.distributed.checkpoint as hf_dist_cp
    hf_dist_cp._check_distributed_checkpointing_available = lambda: False
except Exception:
    pass

from huggingface_hub import HfApi, login, snapshot_download, hf_hub_download
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=True)
    print("✓ Hugging Face authentication successful.")
else:
    print("⚠ Running without Hugging Face login; authenticated pushes will be disabled.")


In [ ]:
# 2. Execution Profile Selection & Global Configuration
# Toggle PROFILE between "SMOKE" (fast syntax & flow test) and "CORE" (full production training)
PROFILE = "CORE"  # Options: "SMOKE", "CORE"

PROFILES = {
    "SMOKE": {
        "max_train_queries": 20,       # 20 queries for quick smoke verification
        "max_candidates": 100,         # 100 candidate cases
        "max_dev_queries": 15,         # 15 dev queries
        "max_test_queries": 15,        # 15 test queries
        "train_epochs": 1,
        "batch_size": 4,
        "eval_steps": 20,
        "save_steps": 20,
        "warmup_steps": 10,
        "max_chunks": 4,               # 4 chunks x 512 tokens
        "chunk_size": 512,
        "chunk_stride": 256,
        "num_poly_codes": 8,           # M=8 aspect vectors
        "description": "Fast Functional Verification (Smoke Mode)"
    },
    "CORE": {
        "max_train_queries": None,     # Full dataset
        "max_candidates": None,        # Full candidate pool
        "max_dev_queries": None,       # Full dev validation
        "max_test_queries": None,      # Full test set
        "train_epochs": 4,
        "batch_size": 16,              # Efficient batch size on Dual T4 GPUs
        "eval_steps": 200,
        "save_steps": 200,             # Continuous cloud sync every 200 steps
        "warmup_steps": 500,
        "max_chunks": 8,               # 8 chunks x 512 tokens
        "chunk_size": 512,
        "chunk_stride": 256,
        "num_poly_codes": 8,           # M=8 aspect vectors
        "description": "Full Production Multi-GPU Training (Core Mode)"
    }
}

CFG = PROFILES[PROFILE]
print(f"=== Active Profile: {PROFILE} ({CFG['description']}) ===")

# Global Constants & Paths
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_MASTER_REPO = "sarweshbarnwal/CS781-Legal-Models"
BASE_MODEL_SUBFOLDER = "model4-capacity-boundary-154m"
EXPERIMENT_NAME = "sub07_model4_poly_encoder_multiaspect"
HUB_SUBFOLDER = f"Phase_7/{EXPERIMENT_NAME}"

OUTPUT_DIR = "/kaggle/working/phase7_sub07" if os.path.exists("/kaggle") else "./phase7_sub07"
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")
LOSS_PLOT_PATH = os.path.join(OUTPUT_DIR, "loss_and_metrics_plot.png")
METRICS_JSON_PATH = os.path.join(OUTPUT_DIR, "val_metrics.json")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Local artifacts directory: {OUTPUT_DIR}")
print(f"HF Hub destination: {HF_MASTER_REPO}/{HUB_SUBFOLDER}")


In [ ]:
# 3. Hardware Acceleration & Dual-T4 Multi-GPU Setup
print("=== Hardware Acceleration Verification ===")
gpu_count = torch.cuda.device_count()
print(f"CUDA Available: {torch.cuda.is_available()} | GPU Count: {gpu_count}")

if torch.cuda.is_available():
    primary_device = torch.device("cuda:0")
    for i in range(gpu_count):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU [{i}]: {props.name} | Total VRAM: {props.total_memory / (1024**3):.2f} GB")
    
    torch.backends.cuda.enable_mem_efficient_sdp(True)
    torch.backends.cuda.enable_math_sdp(True)
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cudnn.benchmark = True
    print("✓ PyTorch 2.0 SDPA memory-efficient attention activated.")
else:
    primary_device = torch.device("cpu")
    print("⚠ No GPU detected. Running on CPU fallback.")

device = primary_device


In [ ]:
# 4. Dataset Ingestion & Preprocessing (Exploration-Lab/CS-781-Capstone)
import json

raw_data_dir = os.path.join(OUTPUT_DIR, "raw_data")
os.makedirs(raw_data_dir, exist_ok=True)

print("Downloading Phase 7 PCR dataset from Hugging Face...")
snapshot_download(
    repo_id=HF_DATASET_REPO,
    repo_type="dataset",
    local_dir=raw_data_dir,
    allow_patterns=["Phase_7_PCR/*"],
    token=HF_TOKEN
)

pcr_base = os.path.join(raw_data_dir, "Phase_7_PCR")
if not os.path.exists(pcr_base):
    for root, dirs, files in os.walk(raw_data_dir):
        if "train.json" in files:
            pcr_base = root
            break

print(f"PCR Data Directory: {pcr_base}")

def load_json_file(filename: str) -> dict:
    candidates = [
        os.path.join(pcr_base, filename),
        os.path.join(pcr_base, "Train", filename),
        os.path.join(pcr_base, "Dev", filename),
        os.path.join(pcr_base, "Test", filename),
    ]
    for path in candidates:
        if os.path.exists(path):
            with open(path, "r", encoding="utf-8") as f:
                return json.load(f)
    for root, _, files in os.walk(pcr_base):
        if filename in files:
            with open(os.path.join(root, filename), "r", encoding="utf-8") as f:
                return json.load(f)
    raise FileNotFoundError(f"Could not locate {filename} under {pcr_base}")

train_queries_raw = load_json_file("train.json")
train_candidates_raw = load_json_file("train_candidates.json")
dev_queries_raw = load_json_file("dev.json")
dev_candidates_raw = load_json_file("dev_candidates.json")
test_queries_raw = load_json_file("test.json")
test_candidates_raw = load_json_file("test_candidates.json")

print(f"Loaded Train Queries: {len(train_queries_raw):,} | Train Candidates: {len(train_candidates_raw):,}")
print(f"Loaded Dev Queries:   {len(dev_queries_raw):,} | Dev Candidates:   {len(dev_candidates_raw):,}")
print(f"Loaded Test Queries:  {len(test_queries_raw):,} | Test Candidates:  {len(test_candidates_raw):,}")

def flatten_case_text(case_obj: Any) -> str:
    if isinstance(case_obj, str):
        return case_obj.strip()
    elif isinstance(case_obj, list):
        return " ".join([str(x).strip() for x in case_obj if str(x).strip()])
    elif isinstance(case_obj, dict):
        if "text" in case_obj:
            return flatten_case_text(case_obj["text"])
        vals = [flatten_case_text(v) for v in case_obj.values() if isinstance(v, (str, list))]
        return " ".join(vals)
    return str(case_obj).strip()

def process_query_dict(q_dict: dict, max_count: Optional[int] = None) -> List[dict]:
    items = []
    for qid, qdata in list(q_dict.items())[:max_count]:
        if isinstance(qdata, dict):
            text = flatten_case_text(qdata.get("text", qdata.get("query_text", "")))
            gold_cids = qdata.get("relevant_candidates", qdata.get("gold_candidates", []))
            if isinstance(gold_cids, str):
                gold_cids = [gold_cids]
        else:
            text = flatten_case_text(qdata)
            gold_cids = []
        items.append({"qid": str(qid), "text": text, "gold_cids": [str(c) for c in gold_cids]})
    return items

def process_candidate_dict(c_dict: dict, max_count: Optional[int] = None) -> Dict[str, str]:
    processed = {}
    for cid, cdata in list(c_dict.items())[:max_count]:
        processed[str(cid)] = flatten_case_text(cdata)
    return processed

def process_splits_aligned(q_dict: dict, c_dict: dict, max_queries: Optional[int], max_cands: Optional[int]):
    queries = process_query_dict(q_dict, max_queries)
    if max_cands is None:
        candidates = {str(cid): flatten_case_text(cdata) for cid, cdata in c_dict.items()}
        return queries, candidates

    gold_cids = {cid for q in queries for cid in q["gold_cids"] if cid in c_dict or str(cid) in c_dict}
    selected_cids = [str(cid) for cid in c_dict.keys() if str(cid) in gold_cids]
    for cid in c_dict.keys():
        if len(selected_cids) >= max_cands:
            break
        if str(cid) not in gold_cids:
            selected_cids.append(str(cid))

    if len(selected_cids) < 7:
        for cid in c_dict.keys():
            if str(cid) not in selected_cids:
                selected_cids.append(str(cid))
            if len(selected_cids) >= min(7, len(c_dict)):
                break

    candidates = {cid: flatten_case_text(c_dict[cid]) for cid in selected_cids}
    return queries, candidates

train_queries, train_candidates = process_splits_aligned(train_queries_raw, train_candidates_raw, CFG["max_train_queries"], CFG["max_candidates"])
dev_queries, dev_candidates = process_splits_aligned(dev_queries_raw, dev_candidates_raw, CFG["max_dev_queries"], CFG["max_candidates"])
test_queries, test_candidates = process_splits_aligned(test_queries_raw, test_candidates_raw, CFG["max_test_queries"], CFG["max_candidates"])

print(f"Profile: {PROFILE} => Train Queries: {len(train_queries)}, Dev Queries: {len(dev_queries)}, Test Queries: {len(test_queries)}")
print(f"Test Candidates in evaluation pool: {len(test_candidates):,}")


In [ ]:
# Defensive PyTorch & Transformers compatibility guard for Python 3.13 / Kaggle
import sys
import types
import importlib
import torch

try:
    _u = importlib.import_module("torch._utils")
except Exception:
    _u = types.ModuleType("torch._utils")

if not hasattr(_u, "_get_device_index"):
    def _get_device_index(device, optional=False, allow_cpu=False):
        if isinstance(device, str): device = torch.device(device)
        if isinstance(device, torch.device): return -1 if device.type == "cpu" else (device.index or 0)
        if isinstance(device, int): return device
        return 0
    _u._get_device_index = _get_device_index

if not hasattr(_u, "try_import"):
    def try_import(name):
        try:
            return importlib.import_module(name)
        except Exception:
            return None
    _u.try_import = try_import

def _fallback_getattr(name):
    if name == "try_import":
        return _u.try_import
    if name == "_get_device_index":
        return _u._get_device_index
    return lambda *args, **kwargs: None

_u.__getattr__ = _fallback_getattr
setattr(torch, "_utils", _u)
sys.modules["torch._utils"] = _u

try:
    import transformers.utils.import_utils as hf_import_utils
    hf_import_utils.is_torchvision_available = lambda: False
except Exception:
    pass

try:
    import transformers.distributed.checkpoint as hf_dist_cp
    hf_dist_cp._check_distributed_checkpointing_available = lambda: False
except Exception:
    pass

# 5. Tokenizer Initialization from Model 4 (LLaMA-style 154M)
from transformers import PreTrainedTokenizerFast

local_tok_dir = os.path.join(OUTPUT_DIR, "tokenizer")
os.makedirs(local_tok_dir, exist_ok=True)

try:
    print(f"Attempting to download tokenizer from {HF_MASTER_REPO}...")
    snapshot_download(
        repo_id=HF_MASTER_REPO,
        allow_patterns=[f"{BASE_MODEL_SUBFOLDER}/*token*"],
        local_dir=local_tok_dir,
        token=HF_TOKEN
    )
except Exception as e:
    print(f"Specific tokenizer pattern notice: {e}. Downloading entire model subfolder...")
    snapshot_download(
        repo_id=HF_MASTER_REPO,
        allow_patterns=[f"{BASE_MODEL_SUBFOLDER}/*"],
        local_dir=local_tok_dir,
        token=HF_TOKEN
    )

tok_json_path = os.path.join(local_tok_dir, BASE_MODEL_SUBFOLDER, "tokenizer", "tokenizer.json")
if not os.path.exists(tok_json_path):
    tok_json_path = os.path.join(local_tok_dir, BASE_MODEL_SUBFOLDER, "tokenizer.json")
if not os.path.exists(tok_json_path):
    tok_json_path = os.path.join(local_tok_dir, "tokenizer.json")

tokenizer = PreTrainedTokenizerFast(
    tokenizer_file=tok_json_path,
    bos_token="<bos>",
    eos_token="<eos>",
    unk_token="<unk>",
    pad_token="<pad>"
)
tokenizer.padding_side = "right"
print(f"✓ Tokenizer loaded successfully. Vocab Size: {len(tokenizer):,}")


In [ ]:
# 6. Poly-Encoder Multi-Vector Late Interaction Architecture (Humeau et al., ICLR 2020)
# Defensive PyTorch & Transformers compatibility guard for Python 3.13 / Kaggle
import sys
import types
import importlib
import torch

try:
    _u = importlib.import_module("torch._utils")
except Exception:
    _u = types.ModuleType("torch._utils")

if not hasattr(_u, "_get_device_index"):
    def _get_device_index(device, optional=False, allow_cpu=False):
        if isinstance(device, str): device = torch.device(device)
        if isinstance(device, torch.device): return -1 if device.type == "cpu" else (device.index or 0)
        if isinstance(device, int): return device
        return 0
    _u._get_device_index = _get_device_index

if not hasattr(_u, "try_import"):
    def try_import(name):
        try:
            return importlib.import_module(name)
        except Exception:
            return None
    _u.try_import = try_import

def _fallback_getattr(name):
    if name == "try_import":
        return _u.try_import
    if name == "_get_device_index":
        return _u._get_device_index
    return lambda *args, **kwargs: None

_u.__getattr__ = _fallback_getattr
setattr(torch, "_utils", _u)
sys.modules["torch._utils"] = _u

try:
    import transformers.utils.import_utils as hf_import_utils
    hf_import_utils.is_torchvision_available = lambda: False
except Exception:
    pass

try:
    import transformers.distributed.checkpoint as hf_dist_cp
    hf_dist_cp._check_distributed_checkpointing_available = lambda: False
except Exception:
    pass

from transformers import LlamaConfig, LlamaModel

class PolyEncoderQueryPooler(nn.Module):
    """
    Poly-Encoder Query Multi-Vector Pooler:
    Extracts M=8 aspect vectors from chunk representations using learnable codebook vectors:
        w_{m} = Softmax((c_m * H^T) / sqrt(d))
        y_m = sum_i (w_{m, i} * h_i)
    Each codebook vector extracts a different semantic aspect of the query case.
    """
    def __init__(self, hidden_size: int = 896, num_codes: int = 8):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_codes = num_codes
        # Learnable aspect codebook: [M, D]
        self.codebook = nn.Parameter(torch.randn(num_codes, hidden_size) * 0.02)
        self.layer_norm = nn.LayerNorm(hidden_size)

    def forward(self, chunk_reprs: torch.Tensor, chunk_mask: torch.Tensor) -> torch.Tensor:
        """
        chunk_reprs: [Batch, K, D]
        chunk_mask:  [Batch, K]
        Returns:
            aspect_vectors: [Batch, M, D]
        """
        B, K, D = chunk_reprs.shape
        M = self.num_codes

        # Expand codebook across batch: [B, M, D]
        codes = self.codebook.unsqueeze(0).expand(B, M, D)

        # Attention scores between codes and chunks: [B, M, K]
        scores = torch.bmm(codes, chunk_reprs.transpose(1, 2)) / math.sqrt(D)

        # Mask inactive chunks
        mask_expanded = chunk_mask.unsqueeze(1).expand(B, M, K)  # [B, M, K]
        scores = scores.masked_fill(mask_expanded == 0, -1e4)

        weights = F.softmax(scores, dim=-1)  # [B, M, K]
        aspect_vectors = torch.bmm(weights, chunk_reprs)  # [B, M, D]
        aspect_vectors = self.layer_norm(aspect_vectors)
        return F.normalize(aspect_vectors, p=2, dim=-1)


class CandidateDocPooler(nn.Module):
    """
    Compresses candidate document chunks into a single L2-normalized dense vector v in R^D.
    Preserves fast offline pre-computation and sub-millisecond retrieval.
    """
    def __init__(self, hidden_size: int = 896):
        super().__init__()
        self.layer_norm = nn.LayerNorm(hidden_size)

    def forward(self, chunk_reprs: torch.Tensor, chunk_mask: torch.Tensor) -> torch.Tensor:
        mask_expanded = chunk_mask.unsqueeze(-1).expand_as(chunk_reprs).float()
        sum_hidden = torch.sum(chunk_reprs * mask_expanded, dim=1)
        sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
        doc_hidden = self.layer_norm(sum_hidden / sum_mask)
        return F.normalize(doc_hidden, p=2, dim=-1)


class LegalPolyEncoder(nn.Module):
    """
    Full Poly-Encoder Two-Tower System:
    - Backbone: LLaMA-style Model 4 (14 layers, d=896)
    - Query Tower: Generates M=8 aspect vectors [B, M, D]
    - Candidate Tower: Generates single vector [B, D]
    - Late-Interaction Score:
        alpha_m = Softmax((v * y_m) / sqrt(d))
        u(v) = sum_m (alpha_m * y_m)
        score(q, v) = u(v) * v
    """
    def __init__(self, backbone: LlamaModel, hidden_size: int = 896, max_chunks: int = 8, num_codes: int = 8):
        super().__init__()
        self.backbone = backbone
        self.hidden_size = hidden_size
        self.max_chunks = max_chunks
        self.num_codes = num_codes
        self.query_pooler = PolyEncoderQueryPooler(hidden_size=hidden_size, num_codes=num_codes)
        self.cand_pooler = CandidateDocPooler(hidden_size=hidden_size)

    def encode_backbone_chunks(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        B, K, L = input_ids.shape
        flat_ids = input_ids.view(B * K, L)
        flat_mask = attention_mask.view(B * K, L)

        chunk_reprs_list = []
        max_sub_batch = 4
        for s_idx in range(0, flat_ids.size(0), max_sub_batch):
            e_idx = min(s_idx + max_sub_batch, flat_ids.size(0))
            sub_ids = flat_ids[s_idx:e_idx]
            sub_mask = flat_mask[s_idx:e_idx]

            sub_out = self.backbone(input_ids=sub_ids, attention_mask=sub_mask)
            sub_hidden = sub_out.last_hidden_state

            mask_expanded = sub_mask.unsqueeze(-1).expand_as(sub_hidden).float()
            sum_hidden = torch.sum(sub_hidden * mask_expanded, dim=1)
            sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
            sub_repr = sum_hidden / sum_mask
            chunk_reprs_list.append(sub_repr)

        chunk_reprs = torch.cat(chunk_reprs_list, dim=0)
        return chunk_reprs.view(B, K, self.hidden_size)

    def encode_queries(self, q_ids: torch.Tensor, q_attn: torch.Tensor, q_cmask: torch.Tensor) -> torch.Tensor:
        # Returns [B, M, D]
        chunk_reprs = self.encode_backbone_chunks(q_ids, q_attn)
        return self.query_pooler(chunk_reprs, q_cmask)

    def encode_candidates(self, d_ids: torch.Tensor, d_attn: torch.Tensor, d_cmask: torch.Tensor) -> torch.Tensor:
        # Returns [B, D]
        chunk_reprs = self.encode_backbone_chunks(d_ids, d_attn)
        return self.cand_pooler(chunk_reprs, d_cmask)

    def score_late_interaction(self, query_aspects: torch.Tensor, cand_vectors: torch.Tensor) -> torch.Tensor:
        """
        Computes pairwise similarity matrix between queries and candidates:
        query_aspects: [B_q, M, D]
        cand_vectors:  [B_c, D]
        Returns:
            scores: [B_q, B_c]
        """
        B_q, M, D = query_aspects.shape
        B_c, _ = cand_vectors.shape

        # Dot product between all query aspects and candidate vectors: [B_q, M, B_c]
        # query_aspects: [B_q, M, D] -> cand_vectors.T: [D, B_c] => [B_q, M, B_c]
        aspect_cand_scores = torch.einsum("bmd,cd->bmc", query_aspects, cand_vectors) / math.sqrt(D)

        # Softmax over the M aspects for each (query, candidate) pair
        # weights: [B_q, M, B_c]
        weights = F.softmax(aspect_cand_scores, dim=1)

        # Weighted combination of aspects for each candidate: u(v) = sum_m (weights_m * y_m)
        # u(v): [B_q, B_c, D]
        # weights.permute(0, 2, 1): [B_q, B_c, M]
        u_v = torch.einsum("bcm,bmd->bcd", weights.permute(0, 2, 1), query_aspects)

        # Final score: u(v) * v for each pair: [B_q, B_c]
        scores = torch.einsum("bcd,cd->bc", u_v, cand_vectors)
        return scores

    def forward(self, q_ids, q_attn, q_cmask, d_ids=None, d_attn=None, d_cmask=None):
        q_aspects = self.encode_queries(q_ids, q_attn, q_cmask)  # [B, M, D]
        if d_ids is not None:
            c_vecs = self.encode_candidates(d_ids, d_attn, d_cmask)  # [B, D]
            return q_aspects, c_vecs
        return q_aspects


print("Downloading Model 4 (154M) base weights from Master Hugging Face Hub...")
m4_dir = os.path.join(OUTPUT_DIR, "model4_base")
os.makedirs(m4_dir, exist_ok=True)

snapshot_download(
    repo_id=HF_MASTER_REPO,
    allow_patterns=[f"{BASE_MODEL_SUBFOLDER}/*"],
    local_dir=m4_dir,
    token=HF_TOKEN
)

base_subpath = os.path.join(m4_dir, BASE_MODEL_SUBFOLDER)
print(f"Loading base LlamaModel from: {base_subpath}")
base_llama = LlamaModel.from_pretrained(
    base_subpath,
    torch_dtype=torch.float32,
    device_map=None
)
base_llama.gradient_checkpointing_enable()
print("✓ Gradient checkpointing enabled on base LLaMA backbone.")

model = LegalPolyEncoder(
    backbone=base_llama,
    hidden_size=base_llama.config.hidden_size,
    max_chunks=CFG["max_chunks"],
    num_codes=CFG["num_poly_codes"]
)
model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"=== Model Parameter Accounting ===")
print(f"  Total Parameters:     {total_params:,}")
print(f"  Trainable Parameters: {trainable_params:,} ({trainable_params / total_params * 100:.2f}%)")
print(f"  Poly Aspect Codes:    {CFG['num_poly_codes']} vectors in R^{base_llama.config.hidden_size}")


In [ ]:
# 7. Poly-Encoder InfoNCE Ranking Loss (tau=0.05) & Robust Data Pipeline

class PolyEncoderInfoNCELoss(nn.Module):
    """
    InfoNCE Loss over Poly-Encoder Multi-Vector Late Interaction Scores:
    For a batch of B queries and B candidate cases:
      scores = score_late_interaction(q_aspects, cand_vecs) / tau  # [B, B]
      loss = CrossEntropyLoss(scores, arange(B))
    """
    def __init__(self, tau: float = 0.05):
        super().__init__()
        self.tau = tau

    def forward(self, model: LegalPolyEncoder, q_aspects: torch.Tensor, cand_vecs: torch.Tensor) -> torch.Tensor:
        B = q_aspects.shape[0]
        if B <= 1:
            return torch.tensor(0.0, device=q_aspects.device, requires_grad=True)

        # Compute late interaction matrix: [B, B]
        if hasattr(model, "module"):
            sims = model.module.score_late_interaction(q_aspects, cand_vecs)
        else:
            sims = model.score_late_interaction(q_aspects, cand_vecs)

        logits = sims / self.tau
        targets = torch.arange(B, device=logits.device)
        return F.cross_entropy(logits, targets)


poly_criterion = PolyEncoderInfoNCELoss(tau=0.05)


def chunk_document(
    text: str,
    tokenizer,
    max_chunks: int = 8,
    chunk_size: int = 512,
    chunk_stride: int = 256
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    enc = tokenizer(
        text,
        truncation=False,
        add_special_tokens=True,
        return_attention_mask=True
    )
    all_input_ids = enc["input_ids"]
    all_attn_mask = enc["attention_mask"]
    total_len = len(all_input_ids)

    chunks_ids = []
    chunks_mask = []

    if total_len <= chunk_size:
        pad_len = chunk_size - total_len
        chunks_ids.append(all_input_ids + [tokenizer.pad_token_id] * pad_len)
        chunks_mask.append(all_attn_mask + [0] * pad_len)
    else:
        start = 0
        while start < total_len and len(chunks_ids) < max_chunks:
            end = min(start + chunk_size, total_len)
            c_ids = all_input_ids[start:end]
            c_mask = all_attn_mask[start:end]
            if len(c_ids) < chunk_size:
                pad_len = chunk_size - len(c_ids)
                c_ids = c_ids + [tokenizer.pad_token_id] * pad_len
                c_mask = c_mask + [0] * pad_len
            chunks_ids.append(c_ids)
            chunks_mask.append(c_mask)
            if end >= total_len:
                break
            start += chunk_stride

    num_actual = len(chunks_ids)
    while len(chunks_ids) < max_chunks:
        chunks_ids.append([tokenizer.pad_token_id] * chunk_size)
        chunks_mask.append([0] * chunk_size)

    chunk_mask = [1] * num_actual + [0] * (max_chunks - num_actual)

    return (
        torch.tensor(chunks_ids, dtype=torch.long),
        torch.tensor(chunks_mask, dtype=torch.long),
        torch.tensor(chunk_mask, dtype=torch.long)
    )


class LegalPCRDataset(Dataset):
    def __init__(self, queries: List[dict], candidates: Dict[str, str], tokenizer, cfg: dict):
        self.tokenizer = tokenizer
        self.cfg = cfg
        self.samples = []
        cand_keys = list(candidates.keys())

        for q in queries:
            qid = q["qid"]
            q_text = q["text"]
            gold_cids = q.get("gold_cids", [])
            valid_golds = [cid for cid in gold_cids if cid in candidates]

            if valid_golds:
                for g_cid in valid_golds:
                    self.samples.append({
                        "qid": qid,
                        "query_text": q_text,
                        "cand_id": g_cid,
                        "cand_text": candidates[g_cid]
                    })
            elif cand_keys:
                # Guaranteed fallback for SMOKE profile
                self.samples.append({
                    "qid": qid,
                    "query_text": q_text,
                    "cand_id": cand_keys[0],
                    "cand_text": candidates[cand_keys[0]]
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        q_ids, q_attn, q_cmask = chunk_document(
            s["query_text"], self.tokenizer,
            max_chunks=self.cfg["max_chunks"],
            chunk_size=self.cfg["chunk_size"],
            chunk_stride=self.cfg["chunk_stride"]
        )
        d_ids, d_attn, d_cmask = chunk_document(
            s["cand_text"], self.tokenizer,
            max_chunks=self.cfg["max_chunks"],
            chunk_size=self.cfg["chunk_size"],
            chunk_stride=self.cfg["chunk_stride"]
        )
        return {
            "q_ids": q_ids, "q_attn": q_attn, "q_cmask": q_cmask,
            "d_ids": d_ids, "d_attn": d_attn, "d_cmask": d_cmask,
            "qid": s["qid"], "cand_id": s["cand_id"]
        }


def collate_pcr_batch(batch):
    return {
        "q_ids": torch.stack([b["q_ids"] for b in batch]),
        "q_attn": torch.stack([b["q_attn"] for b in batch]),
        "q_cmask": torch.stack([b["q_cmask"] for b in batch]),
        "d_ids": torch.stack([b["d_ids"] for b in batch]),
        "d_attn": torch.stack([b["d_attn"] for b in batch]),
        "d_cmask": torch.stack([b["d_cmask"] for b in batch]),
        "qids": [b["qid"] for b in batch],
        "cand_ids": [b["cand_id"] for b in batch],
    }

train_dataset = LegalPCRDataset(train_queries, train_candidates, tokenizer, CFG)
train_loader = DataLoader(
    train_dataset,
    batch_size=CFG["batch_size"],
    shuffle=True,
    num_workers=0,
    collate_fn=collate_pcr_batch,
    drop_last=(len(train_dataset) >= CFG["batch_size"] and PROFILE != "SMOKE")
)

print(f"✓ Train DataLoader assembled: {len(train_dataset):,} samples, {len(train_loader)} batches/epoch.")


In [ ]:
# 8. Pre-flight Assertion: Verify Poly-Encoder Late Interaction & Gradient Flow
print("=== Running Pre-flight Poly-Encoder Late Interaction Assertion ===")
model.train()
assert len(train_loader) > 0, f"DataLoader has 0 batches! Dataset size: {len(train_dataset)}, batch_size: {CFG['batch_size']}"
sample_batch = next(iter(train_loader))

# Micro-slice to at most 2 samples to verify Poly-Encoder flow safely
q_ids = sample_batch["q_ids"][:1].to(device)
q_attn = sample_batch["q_attn"][:2].to(device)
q_cmask = sample_batch["q_cmask"][:2].to(device)
d_ids = sample_batch["d_ids"][:1].to(device)
d_attn = sample_batch["d_attn"][:2].to(device)
d_cmask = sample_batch["d_cmask"][:2].to(device)

model.zero_grad()
with torch.amp.autocast("cuda", dtype=torch.float16):
    q_aspects, c_vecs = model(q_ids, q_attn, q_cmask, d_ids, d_attn, d_cmask)

# Check shapes: q_aspects: [B, M, D], c_vecs: [B, D]
B = q_ids.shape[0]
M = CFG["num_poly_codes"]
D = model.hidden_size
assert q_aspects.shape == (B, M, D), f"ERROR: Query aspects shape mismatch: {q_aspects.shape} vs ({B}, {M}, {D})"
assert c_vecs.shape == (B, D), f"ERROR: Candidate vectors shape mismatch: {c_vecs.shape} vs ({B}, {D})"

with torch.amp.autocast("cuda", dtype=torch.float16):
    loss = poly_criterion(model, q_aspects, c_vecs)
loss.backward()

assert not torch.isnan(loss), "ERROR: Preflight loss is NaN!"
assert loss.item() > 0.0, f"ERROR: Preflight loss non-positive: {loss.item()}"

# Verify non-zero gradients on codebook vectors
codebook_grad = model.query_pooler.codebook.grad
assert codebook_grad is not None and codebook_grad.abs().sum().item() > 0, "ERROR: Codebook vectors received ZERO gradients!"

# Verify non-zero gradients on backbone parameters
backbone_has_grad = any(p.grad is not None and p.grad.abs().sum().item() > 0 for p in model.backbone.parameters())
assert backbone_has_grad, "ERROR: Model 4 backbone received ZERO gradients!"

print(f"✓ Pre-flight Assertion PASSED: InfoNCE loss = {loss.item():.4f}")
print(f"✓ Poly-Encoder Aspect Codebook ({M} codes) & Backbone successfully receiving gradients.")
model.zero_grad()


In [ ]:
# 9. Continuous Cloud Sync Callback & Automatic Crash Recovery Engine
from huggingface_hub import list_repo_files

class HfContinuousSyncCallback:
    """
    Threaded, non-blocking cloud synchronization of checkpoints to Hugging Face Hub.
    Ensures that every checkpoint-{step} is persisted without delaying the training loop.
    """
    def __init__(self, repo_id: str, subfolder: str, token: Optional[str] = None):
        self.repo_id = repo_id
        self.subfolder = subfolder
        self.token = token
        self.api = HfApi()

    def sync_checkpoint_async(self, checkpoint_dir: str, step: int):
        if not self.token:
            print("⚠ HF_TOKEN not set; skipping remote sync.")
            return

        def _upload_task():
            try:
                ckpt_folder_name = os.path.basename(checkpoint_dir)
                remote_path = f"{self.subfolder}/{ckpt_folder_name}"
                print(f"[CloudSync] Starting upload for step {step} -> {self.repo_id}/{remote_path}...")
                self.api.upload_folder(
                    folder_path=checkpoint_dir,
                    path_in_repo=remote_path,
                    repo_id=self.repo_id,
                    token=self.token
                )
                print(f"[CloudSync] ✓ Step {step} successfully pushed to Hugging Face.")
            except Exception as e:
                print(f"[CloudSync] ⚠ Warning: Cloud sync failed for step {step}: {e}")

        thread = threading.Thread(target=_upload_task, daemon=True)
        thread.start()


sync_callback = HfContinuousSyncCallback(
    repo_id=HF_MASTER_REPO,
    subfolder=HUB_SUBFOLDER,
    token=HF_TOKEN
)


def find_latest_checkpoint(local_ckpt_dir: str, hf_repo: str, hf_subfolder: str) -> Tuple[Optional[str], int]:
    """
    Finds the latest available checkpoint locally, or checks remote HF Hub repo if local is empty.
    """
    local_ckpts = glob.glob(os.path.join(local_ckpt_dir, "checkpoint-*"))
    max_step = -1
    best_ckpt = None

    for c in local_ckpts:
        try:
            step = int(os.path.basename(c).replace("checkpoint-", ""))
            if step > max_step:
                max_step = step
                best_ckpt = c
        except Exception:
            pass

    if best_ckpt:
        print(f"✓ Found local checkpoint: {best_ckpt} (Step {max_step})")
        return best_ckpt, max_step

    # Check remote Hugging Face Hub
    if HF_TOKEN:
        try:
            print(f"Querying remote repo {hf_repo} under {hf_subfolder} for checkpoints...")
            files = list_repo_files(repo_id=hf_repo, token=HF_TOKEN)
            remote_steps = []
            for f in files:
                if f.startswith(hf_subfolder) and "checkpoint-" in f:
                    parts = f.split("/")
                    for p in parts:
                        if p.startswith("checkpoint-"):
                            try:
                                s = int(p.replace("checkpoint-", ""))
                                remote_steps.append((s, p))
                            except Exception:
                                pass
            if remote_steps:
                remote_steps.sort(key=lambda x: x[0], reverse=True)
                highest_step, folder_name = remote_steps[0]
                print(f"Found remote checkpoint at Step {highest_step}. Downloading...")
                dl_dir = os.path.join(local_ckpt_dir, folder_name)
                os.makedirs(dl_dir, exist_ok=True)
                snapshot_download(
                    repo_id=hf_repo,
                    allow_patterns=[f"{hf_subfolder}/{folder_name}/*"],
                    local_dir=local_ckpt_dir,
                    token=HF_TOKEN
                )
                nested = os.path.join(local_ckpt_dir, hf_subfolder, folder_name)
                if os.path.exists(nested):
                    best_ckpt = nested
                else:
                    best_ckpt = dl_dir
                return best_ckpt, highest_step
        except Exception as e:
            print(f"Remote checkpoint check failed: {e}")

    return None, 0


In [ ]:
# 10. Multi-GPU Training Loop with Poly-Encoder & Dual-T4 Acceleration
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
scaler = torch.cuda.amp.GradScaler()

total_steps = CFG["train_epochs"] * len(train_loader)
scheduler = CosineAnnealingLR(optimizer, T_max=max(total_steps, 1), eta_min=1e-6)

raw_model = model
if torch.cuda.device_count() > 1:
    print(f"✓ Parallelizing model forward pass across {torch.cuda.device_count()} GPUs using DataParallel.")
    train_model = nn.DataParallel(raw_model)
else:
    train_model = raw_model

# Check for existing checkpoint to resume
latest_ckpt, resume_step = find_latest_checkpoint(CHECKPOINT_DIR, HF_MASTER_REPO, HUB_SUBFOLDER)
start_step = 0
best_f1 = 0.0

if latest_ckpt and os.path.exists(os.path.join(latest_ckpt, "model_weights.pt")):
    try:
        print(f"Resuming training from checkpoint: {latest_ckpt} at step {resume_step}...")
        raw_model.load_state_dict(torch.load(os.path.join(latest_ckpt, "model_weights.pt"), map_location=device))
        opt_path = os.path.join(latest_ckpt, "optimizer.pt")
        if os.path.exists(opt_path):
            optimizer.load_state_dict(torch.load(opt_path, map_location=device))
        sched_path = os.path.join(latest_ckpt, "scheduler.pt")
        if os.path.exists(sched_path):
            scheduler.load_state_dict(torch.load(sched_path, map_location=device))
        state_info_path = os.path.join(latest_ckpt, "state.json")
        if os.path.exists(state_info_path):
            with open(state_info_path, "r") as sf:
                s_dict = json.load(sf)
                best_f1 = s_dict.get("best_f1", 0.0)
        start_step = resume_step
        print(f"✓ Resumed successfully from step {start_step}. Previous Best F1: {best_f1:.4f}")
    except Exception as e:
        print(f"⚠ Could not restore checkpoint state cleanly: {e}. Starting from step 0.")
        start_step = 0


def evaluate_poly_dev(eval_model, queries: List[dict], candidates: Dict[str, str], tokenizer, cfg: dict) -> Tuple[float, float, float]:
    """
    Evaluates Micro-F1@7 on the dev set using Poly-Encoder Late Interaction.
    """
    eval_model.eval()
    cand_ids = list(candidates.keys())
    if not cand_ids or not queries:
        return 0.0, 0.0, 0.0

    # 1. Pre-encode candidate pool into dense vectors [N, D]
    cand_vectors = []
    with torch.no_grad():
        c_batch_size = 32
        for i in range(0, len(cand_ids), c_batch_size):
            batch_cids = cand_ids[i:i + c_batch_size]
            b_ids, b_attn, b_cmask = [], [], []
            for cid in batch_cids:
                c_ids, c_attn, c_cmask = chunk_document(
                    candidates[cid], tokenizer,
                    max_chunks=cfg["max_chunks"],
                    chunk_size=cfg["chunk_size"],
                    chunk_stride=cfg["chunk_stride"]
                )
                b_ids.append(c_ids)
                b_attn.append(c_attn)
                b_cmask.append(c_cmask)

            c_ids_t = torch.stack(b_ids).to(device)
            c_attn_t = torch.stack(b_attn).to(device)
            c_cmask_t = torch.stack(b_cmask).to(device)

            if isinstance(eval_model, nn.DataParallel):
                e = eval_model.module.encode_candidates(c_ids_t, c_attn_t, c_cmask_t)
            else:
                e = eval_model.encode_candidates(c_ids_t, c_attn_t, c_cmask_t)
            cand_vectors.append(e)

    cand_matrix = torch.cat(cand_vectors, dim=0)  # [Num_Cands, D] on device

    # 2. Encode queries and perform Late-Interaction Scoring
    total_tp = 0
    total_pred = 0
    total_gold = 0

    actual_eval_model = eval_model.module if isinstance(eval_model, nn.DataParallel) else eval_model

    with torch.no_grad():
        for q in queries:
            gold_set = set(q.get("gold_cids", []))
            if not gold_set:
                continue

            q_ids, q_attn, q_cmask = chunk_document(
                q["text"], tokenizer,
                max_chunks=cfg["max_chunks"],
                chunk_size=cfg["chunk_size"],
                chunk_stride=cfg["chunk_stride"]
            )
            q_ids_t = q_ids.unsqueeze(0).to(device)
            q_attn_t = q_attn.unsqueeze(0).to(device)
            q_cmask_t = q_cmask.unsqueeze(0).to(device)

            # Query aspects: [1, M, D]
            q_aspects = actual_eval_model.encode_queries(q_ids_t, q_attn_t, q_cmask_t)

            # Late interaction against all candidate vectors: [1, Num_Cands]
            sims = actual_eval_model.score_late_interaction(q_aspects, cand_matrix).squeeze(0)  # [Num_Cands]
            topk_idx = torch.topk(sims, k=min(7, len(cand_ids))).indices.tolist()
            pred_cids = [cand_ids[idx] for idx in topk_idx]

            tp = len(set(pred_cids).intersection(gold_set))
            total_tp += tp
            total_pred += len(pred_cids)
            total_gold += len(gold_set)

    prec = total_tp / max(total_pred, 1)
    rec = total_tp / max(total_gold, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-9)
    return f1, prec, rec


# Main Training Loop
train_history = []
global_step = start_step
eval_interval = CFG["eval_steps"]
save_interval = CFG["save_steps"]

print(f"=== Starting Poly-Encoder Training Loop (Target Steps: {total_steps}) ===")
epoch = 0

while global_step < total_steps:
    epoch += 1
    train_model.train()
    epoch_loss = 0.0

    for step_idx, batch in enumerate(train_loader):
        global_step += 1
        if global_step <= start_step:
            continue

        q_ids = batch["q_ids"].to(device)
        q_attn = batch["q_attn"].to(device)
        q_cmask = batch["q_cmask"].to(device)
        d_ids = batch["d_ids"].to(device)
        d_attn = batch["d_attn"].to(device)
        d_cmask = batch["d_cmask"].to(device)

        optimizer.zero_grad()

        with torch.amp.autocast("cuda", dtype=torch.float16):
            q_aspects, c_vecs = train_model(q_ids, q_attn, q_cmask, d_ids, d_attn, d_cmask)
            loss = poly_criterion(train_model, q_aspects, c_vecs)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(raw_model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        loss_val = loss.item()
        epoch_loss += loss_val

        if global_step % 20 == 0 or global_step == 1:
            curr_lr = optimizer.param_groups[0]["lr"]
            print(f"Epoch [{epoch}] | Step [{global_step}/{total_steps}] | Poly-Encoder Loss: {loss_val:.4f} | LR: {curr_lr:.2e}")

        # Evaluation & Checkpointing
        if global_step % eval_interval == 0 or global_step == total_steps:
            val_f1, val_prec, val_rec = evaluate_poly_dev(train_model, dev_queries, dev_candidates, tokenizer, CFG)
            print(f"--- [Validation @ Step {global_step}] ---")
            print(f"    Poly-Encoder Micro-F1@7:        {val_f1:.4f}")
            print(f"    Poly-Encoder Micro-Precision@7: {val_prec:.4f}")
            print(f"    Poly-Encoder Micro-Recall@7:    {val_rec:.4f}")

            train_history.append({
                "step": global_step,
                "loss": loss_val,
                "val_f1": val_f1,
                "val_prec": val_prec,
                "val_rec": val_rec
            })

            if val_f1 > best_f1:
                best_f1 = val_f1
                print(f"    ⭐ New Best Dev Micro-F1@7: {best_f1:.4f}! Saving checkpoint...")

            # Save checkpoint
            if global_step % save_interval == 0 or global_step == total_steps:
                ckpt_dir = os.path.join(CHECKPOINT_DIR, f"checkpoint-{global_step}")
                os.makedirs(ckpt_dir, exist_ok=True)
                
                torch.save(raw_model.state_dict(), os.path.join(ckpt_dir, "model_weights.pt"))
                torch.save(optimizer.state_dict(), os.path.join(ckpt_dir, "optimizer.pt"))
                torch.save(scheduler.state_dict(), os.path.join(ckpt_dir, "scheduler.pt"))
                
                state_info = {
                    "step": global_step,
                    "epoch": epoch,
                    "val_f1": val_f1,
                    "best_f1": best_f1,
                    "loss": loss_val
                }
                with open(os.path.join(ckpt_dir, "state.json"), "w") as sf:
                    json.dump(state_info, sf, indent=2)

                print(f"✓ Saved local checkpoint: {ckpt_dir}")
                # Non-blocking continuous upload to Hugging Face
                sync_callback.sync_checkpoint_async(ckpt_dir, global_step)

            train_model.train()

        if global_step >= total_steps:
            break

print("✓ Poly-Encoder Training completed successfully.")


In [ ]:
# 11. Plot Poly-Encoder Training History & Evaluation Curves
import matplotlib.pyplot as plt

if train_history:
    steps = [h["step"] for h in train_history]
    losses = [h["loss"] for h in train_history]
    f1s = [h["val_f1"] for h in train_history]
    precs = [h["val_prec"] for h in train_history]
    recs = [h["val_rec"] for h in train_history]

    plt.figure(figsize=(14, 5))

    plt.subplot(1, 2, 1)
    plt.plot(steps, losses, marker='o', color='purple', label="Poly-Encoder InfoNCE Loss (tau=0.05)")
    plt.title("Poly-Encoder Multi-Vector Training Loss")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(steps, f1s, marker='s', color='green', label="Dev Micro-F1@7")
    plt.plot(steps, precs, marker='^', color='blue', label="Dev Micro-Precision@7")
    plt.plot(steps, recs, marker='v', color='orange', label="Dev Micro-Recall@7")
    plt.title("Poly-Encoder Retrieval Evaluation @ 7")
    plt.xlabel("Step")
    plt.ylabel("Score")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend()

    plt.tight_layout()
    plt.savefig(LOSS_PLOT_PATH, dpi=300)
    plt.show()
    print(f"✓ Saved training curves to: {LOSS_PLOT_PATH}")

    with open(METRICS_JSON_PATH, "w") as f:
        json.dump(train_history, f, indent=2)
else:
    print("No training steps executed in this run (or already finished).")


In [ ]:
# 12. Test Candidate Pre-Encoding & Poly-Encoder Late Interaction Retrieval
print("=== Generating Test Predictions using Poly-Encoder Late Interaction ===")
raw_model.eval()

test_cids = list(test_candidates.keys())
print(f"Encoding {len(test_cids):,} Test Candidates with Candidate Pooler across GPUs...")

cand_vectors = []
c_batch_size = 32

with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.float16):
    for start_idx in range(0, len(test_cids), c_batch_size):
        batch_keys = test_cids[start_idx:start_idx + c_batch_size]
        b_ids, b_attn, b_cmask = [], [], []
        for ck in batch_keys:
            c_ids, c_attn, c_cmask = chunk_document(
                test_candidates[ck], tokenizer,
                max_chunks=CFG["max_chunks"],
                chunk_size=CFG["chunk_size"],
                chunk_stride=CFG["chunk_stride"]
            )
            b_ids.append(c_ids)
            b_attn.append(c_attn)
            b_cmask.append(c_cmask)

        t_ids = torch.stack(b_ids).to(device)
        t_attn = torch.stack(b_attn).to(device)
        t_cmask = torch.stack(b_cmask).to(device)

        if torch.cuda.device_count() > 1:
            embs = train_model.module.encode_candidates(t_ids, t_attn, t_cmask)
        else:
            embs = raw_model.encode_candidates(t_ids, t_attn, t_cmask)

        cand_vectors.append(embs)

cand_matrix = torch.cat(cand_vectors, dim=0)  # [Total_Candidates, D] on device
print(f"✓ Encoded Candidate Matrix: {cand_matrix.shape}")

# Encode test queries and compute Top 7 candidate IDs per query via late interaction
predictions = {}
print(f"Ranking Top 7 Candidates for {len(test_queries):,} Test Queries via Late Interaction...")

actual_model = raw_model

with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.float16):
    for q in test_queries:
        qid = q["qid"]
        q_ids, q_attn, q_cmask = chunk_document(
            q["text"], tokenizer,
            max_chunks=CFG["max_chunks"],
            chunk_size=CFG["chunk_size"],
            chunk_stride=CFG["chunk_stride"]
        )
        q_ids_t = q_ids.unsqueeze(0).to(device)
        q_attn_t = q_attn.unsqueeze(0).to(device)
        q_cmask_t = q_cmask.unsqueeze(0).to(device)

        # Multi-vector query aspects: [1, M, D]
        q_aspects = actual_model.encode_queries(q_ids_t, q_attn_t, q_cmask_t)

        # Late interaction scoring against all candidates: [1, Total_Candidates]
        sim_scores = actual_model.score_late_interaction(q_aspects, cand_matrix).squeeze(0)  # [Total_Candidates]

        top7_indices = torch.topk(sim_scores, k=min(7, len(test_cids))).indices.tolist()
        top7_cids = [test_cids[i] for i in top7_indices]

        # In case test pool has fewer than 7 candidates
        while len(top7_cids) < 7 and test_cids:
            for fallback_cid in test_cids:
                if fallback_cid not in top7_cids:
                    top7_cids.append(fallback_cid)
                if len(top7_cids) == 7:
                    break

        predictions[qid] = top7_cids[:7]

print(f"✓ Generated predictions for {len(predictions):,} queries using Poly-Encoder.")


In [ ]:
# 13. Strict CodaBench Output Verification, Packaging & Master Hub Sync
print("=== Strict CodaBench Verification ===")

# Save prediction.json
with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, indent=2)
print(f"✓ Saved prediction.json to: {PREDICTION_PATH}")

# Verification 1: File exists and is non-empty
assert os.path.exists(PREDICTION_PATH), "ERROR: prediction.json not found!"
with open(PREDICTION_PATH, "r", encoding="utf-8") as f:
    loaded_preds = json.load(f)

# Verification 2: Every query has exactly 7 unique candidate IDs
test_cids_set = set(test_candidates.keys())
for qid, cids in loaded_preds.items():
    assert len(cids) == 7, f"ERROR: Query {qid} does not have exactly 7 candidates (got {len(cids)})!"
    assert len(set(cids)) == 7, f"ERROR: Query {qid} contains duplicate candidate IDs!"
    for cid in cids:
        assert cid in test_cids_set, f"ERROR: Candidate {cid} for query {qid} is not in test_candidates.json!"

print("✓ CodaBench Assertions PASSED:")
print(f"  - Total Queries Predicted: {len(loaded_preds):,}")
print("  - Exactly 7 unique candidates per query: VERIFIED.")
print("  - All candidates belong to official test_candidates.json: VERIFIED.")

# Package submission.zip with prediction.json directly at root
with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(PREDICTION_PATH, arcname="prediction.json")

print(f"✓ Successfully packaged submission.zip: {SUBMISSION_ZIP}")
with zipfile.ZipFile(SUBMISSION_ZIP, "r") as check_zip:
    namelist = check_zip.namelist()
    assert namelist == ["prediction.json"], f"ERROR: submission.zip root structure invalid: {namelist}"
print("✓ Root packaging verified: 'prediction.json' is at the root level.")

# Save final Poly-Encoder model weights
final_model_dir = os.path.join(OUTPUT_DIR, "final_poly_encoder")
os.makedirs(final_model_dir, exist_ok=True)
torch.save(raw_model.state_dict(), os.path.join(final_model_dir, "poly_encoder_weights.pt"))
print(f"✓ Final Poly-Encoder model saved to: {final_model_dir}")

# Push final artifacts to Hugging Face Hub under Phase_7/sub07_model4_poly_encoder_multiaspect/
if HF_TOKEN:
    try:
        api = HfApi()
        print(f"Uploading final Phase 7 Sub-07 artifacts to {HF_MASTER_REPO}/{HUB_SUBFOLDER}...")
        api.upload_file(
            path_or_fileobj=PREDICTION_PATH,
            path_in_repo=f"{HUB_SUBFOLDER}/prediction.json",
            repo_id=HF_MASTER_REPO,
            token=HF_TOKEN
        )
        api.upload_file(
            path_or_fileobj=SUBMISSION_ZIP,
            path_in_repo=f"{HUB_SUBFOLDER}/submission.zip",
            repo_id=HF_MASTER_REPO,
            token=HF_TOKEN
        )
        if os.path.exists(LOSS_PLOT_PATH):
            api.upload_file(
                path_or_fileobj=LOSS_PLOT_PATH,
                path_in_repo=f"{HUB_SUBFOLDER}/loss_and_metrics_plot.png",
                repo_id=HF_MASTER_REPO,
                token=HF_TOKEN
            )
        api.upload_folder(
            folder_path=final_model_dir,
            path_in_repo=f"{HUB_SUBFOLDER}/final_model",
            repo_id=HF_MASTER_REPO,
            token=HF_TOKEN
        )
        print("✓ All final Phase 7 Sub-07 artifacts uploaded to Hugging Face Master Repository.")
    except Exception as e:
        print(f"⚠ Master Hub upload failed: {e}")
else:
    print("⚠ HF_TOKEN not present. Skipping final master upload.")

print("=== CS781 Phase 7 Sub-07 Execution Complete ===")
